In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

# --- 1. Load Data ---
print("--- 1. Loading Data ---")
RECEIVALS_FILE = "data/kernel/receivals.csv"
PURCHASE_ORDERS_FILE = "data/kernel/purchase_orders.csv"
MAPPING_FILE = "data/prediction_mapping.csv"
MATERIALS_FILE = "data/extended/materials.csv" # <-- NEW FILE

try:
    df_receivals = pd.read_csv(RECEIVALS_FILE)
    df_po = pd.read_csv(PURCHASE_ORDERS_FILE)
    df_mapping = pd.read_csv(MAPPING_FILE)
    df_materials = pd.read_csv(MATERIALS_FILE) # <-- NEW
    print("All data files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: {e}")
    raise

# --- 2. Perform All Cleaning Steps ---
print("\n--- 2. Cleaning Data ---")

# --- Clean Receivals ---
df_receivals['date_arrival'] = pd.to_datetime(df_receivals['date_arrival'], utc=True, errors='coerce')
df_receivals_cleaned = df_receivals.dropna(
    subset=['rm_id', 'product_id', 'purchase_order_id', 'net_weight']
).copy()
df_receivals_cleaned = df_receivals_cleaned[df_receivals_cleaned['net_weight'] > 0].copy()
print(f"df_receivals_cleaned ready. Shape: {df_receivals_cleaned.shape}")

# --- Clean Materials (Our new mapping file) ---
# We only need product_id and rm_id. We'll drop duplicates.
product_to_rm_map = df_materials[['product_id', 'rm_id']].drop_duplicates()
# Handle cases where one product_id might map to multiple rm_ids (take the first one)
product_to_rm_map = product_to_rm_map.drop_duplicates(subset=['product_id'], keep='first')
print(f"product_to_rm_map ready. Shape: {product_to_rm_map.shape}")

# --- Clean Purchase Orders ---
df_po['delivery_date'] = pd.to_datetime(df_po['delivery_date'], errors='coerce', utc=True)
df_po['created_date_time'] = pd.to_datetime(df_po['created_date_time'], errors='coerce', utc=True)
df_po['modified_date_time'] = pd.to_datetime(df_po['modified_date_time'], errors='coerce', utc=True)
df_po_cleaned = df_po.dropna(subset=['unit_id', 'unit']).copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['unit'] == 'KG'].copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['quantity'] > 0].copy()
print(f"df_po_cleaned (before rm_id). Shape: {df_po_cleaned.shape}")

# --- !!! ADDING RM_ID TO PO TABLE !!! ---
df_po_cleaned = pd.merge(
    df_po_cleaned,
    product_to_rm_map,
    on='product_id',
    how='left' # Keep all POs, even if they don't have a map
)
# Drop POs that we couldn't map to an rm_id
df_po_cleaned = df_po_cleaned.dropna(subset=['rm_id'])
print(f"df_po_cleaned (after rm_id). Shape: {df_po_cleaned.shape}")


# --- Clean Prediction Mapping (Just load, don't clean yet) ---
print(f"df_mapping loaded. Shape: {df_mapping.shape}")

# --- 3. Create Validation Split ---
print("\n--- 3. Creating Validation Split ---")
VALIDATION_START_DATE = pd.to_datetime('2024-08-01', utc=True)
train_receivals = df_receivals_cleaned[df_receivals_cleaned['date_arrival'] < VALIDATION_START_DATE].copy()
validation_receivals = df_receivals_cleaned[df_receivals_cleaned['date_arrival'] >= VALIDATION_START_DATE].copy()
print("Training and validation sets created.")
print(f"Training receivals:       {len(train_receivals)}")
print(f"Validation receivals:     {len(validation_receivals)}")

print("\n--- Setup Complete ---")

--- 1. Loading Data ---
All data files loaded successfully.

--- 2. Cleaning Data ---
df_receivals_cleaned ready. Shape: (122383, 10)
product_to_rm_map ready. Shape: (55, 2)
df_po_cleaned (before rm_id). Shape: (33113, 12)
df_po_cleaned (after rm_id). Shape: (31977, 13)
df_mapping loaded. Shape: (30450, 4)

--- 3. Creating Validation Split ---
Training and validation sets created.
Training receivals:       120268
Validation receivals:     2115

--- Setup Complete ---


In [2]:
# --- 1. Inspect prediction_mapping.csv ---
print("--- 1. Inspecting prediction_mapping.csv ---")
print("Head of mapping file:")
print(df_mapping.head())

print("\nInfo of mapping file:")
df_mapping.info()

--- 1. Inspecting prediction_mapping.csv ---
Head of mapping file:
   ID  rm_id forecast_start_date forecast_end_date
0   1    365          2025-01-01        2025-01-02
1   2    365          2025-01-01        2025-01-03
2   3    365          2025-01-01        2025-01-04
3   4    365          2025-01-01        2025-01-05
4   5    365          2025-01-01        2025-01-06

Info of mapping file:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30450 entries, 0 to 30449
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ID                   30450 non-null  int64 
 1   rm_id                30450 non-null  int64 
 2   forecast_start_date  30450 non-null  object
 3   forecast_end_date    30450 non-null  object
dtypes: int64(2), object(2)
memory usage: 951.7+ KB


## Phase 3a: Building the Forecasting Target (Validation `y`)

This is the most critical data transformation of the project. We now know the `prediction_mapping.csv` file defines our task: for each `rm_id`, we must predict the cumulative weight for a series of `forecast_end_date`s.

Our goal is to create a "ground truth" dataframe for our **local validation set** that mimics this structure. We will create a new dataframe (`val_targets_df`) that has one row for every `(rm_id, date)` pair in our validation period (Aug 2024 - Dec 2024).

For each row, we will calculate the **cumulative `net_weight`** received for that `rm_id` from the start of the validation period (2024-08-01) up to and including that row's `date`.

This dataframe will serve as the "ground truth" (the `y` value, or $A_i$ in the formula) for our validation set.

In [3]:
# --- 1. Clean prediction_mapping.csv ---
print("--- 1. Cleaning prediction_mapping.csv ---")
# Now we use the correct column names found in our inspection
df_mapping['forecast_end_date'] = pd.to_datetime(df_mapping['forecast_end_date'], utc=True)
df_mapping['forecast_start_date'] = pd.to_datetime(df_mapping['forecast_start_date'], utc=True)
print("Mapping file dates converted successfully.")
print(f"Mapping 'forecast_end_date' range: {df_mapping['forecast_end_date'].min().date()} to {df_mapping['forecast_end_date'].max().date()}")


# --- 2. Define our Validation "Universe" ---
print("\n--- 2. Defining validation 'universe' ---")
# Get all unique rm_ids from our validation set
validation_rm_ids = validation_receivals['rm_id'].unique()

# Get all unique dates in our validation set
validation_start = validation_receivals['date_arrival'].min().date()
validation_end = validation_receivals['date_arrival'].max().date()
# Create a *complete* set of dates from the start to the end of our validation period
validation_dates = pd.date_range(start=validation_start, end=validation_end, freq='D', tz='UTC')

print(f"Validation Period: {validation_start} to {validation_end}")
print(f"Number of unique rm_ids in validation: {len(validation_rm_ids)}")
print(f"Number of days in validation: {len(validation_dates)}")
print(f"Total rows in our validation target DF: {len(validation_rm_ids) * len(validation_dates)}")

# --- 3. Create the Validation Target DF ---
print("\n--- 3. Creating validation target dataframe (this may take a moment) ---")
# Create all (rm_id, date) combinations
validation_universe = list(itertools.product(validation_rm_ids, validation_dates))
# We name the date column 'forecast_end_date' to match the final test set structure
val_targets_df = pd.DataFrame(validation_universe, columns=['rm_id', 'forecast_end_date'])

# --- 4. Calculate Actual Daily Receivals ---
# Group our validation receivals by rm_id and day
# This gives us the *actual* weight received on that day
daily_actuals = validation_receivals.groupby(
    ['rm_id', validation_receivals['date_arrival'].dt.date]
)['net_weight'].sum().reset_index(name='daily_net_weight')

# Convert the date_arrival (which is just a 'date') back to a full UTC timestamp so we can merge
daily_actuals['forecast_end_date'] = pd.to_datetime(daily_actuals['date_arrival'], utc=True)


# --- 5. Merge Daily Actuals into our Target DF ---
print("--- 4. Merging daily actuals ---")
# Merge the actuals into our full (rm_id, date) universe
val_targets_df = pd.merge(
    val_targets_df,
    daily_actuals[['rm_id', 'forecast_end_date', 'daily_net_weight']],
    on=['rm_id', 'forecast_end_date'],
    how='left'
)
# Fill NaNs with 0 (for days with no receivals)
val_targets_df['daily_net_weight'] = val_targets_df['daily_net_weight'].fillna(0)

# --- 6. Calculate Cumulative Target ---
print("--- 5. Calculating cumulative sum (target variable) ---")
# Sort by rm_id, then date. This is CRITICAL for a correct cumsum.
val_targets_df = val_targets_df.sort_values(by=['rm_id', 'forecast_end_date'])

# Group by rm_id and calculate the cumulative sum of net_weight
val_targets_df['y_cumulative_weight'] = val_targets_df.groupby('rm_id')['daily_net_weight'].cumsum()

# --- 7. Final Check ---
print("--- 6. Final check of the target dataframe ---")
print(val_targets_df.head())
print("\nExample: one rm_id")
# Show one rm_id to see the cumulative sum in action
non_zero_rm_id = val_targets_df[val_targets_df['y_cumulative_weight'] > 0]['rm_id'].iloc[0]
print(val_targets_df[val_targets_df['rm_id'] == non_zero_rm_id].tail())

print("\nValidation target dataframe (y) created successfully.")

--- 1. Cleaning prediction_mapping.csv ---
Mapping file dates converted successfully.
Mapping 'forecast_end_date' range: 2025-01-02 to 2025-05-31

--- 2. Defining validation 'universe' ---
Validation Period: 2024-08-01 to 2024-12-19
Number of unique rm_ids in validation: 50
Number of days in validation: 141
Total rows in our validation target DF: 7050

--- 3. Creating validation target dataframe (this may take a moment) ---
--- 4. Merging daily actuals ---
--- 5. Calculating cumulative sum (target variable) ---
--- 6. Final check of the target dataframe ---
       rm_id         forecast_end_date  daily_net_weight  y_cumulative_weight
5640  2123.0 2024-08-01 00:00:00+00:00               0.0                  0.0
5641  2123.0 2024-08-02 00:00:00+00:00               0.0                  0.0
5642  2123.0 2024-08-03 00:00:00+00:00               0.0                  0.0
5643  2123.0 2024-08-04 00:00:00+00:00               0.0                  0.0
5644  2123.0 2024-08-05 00:00:00+00:00        

## Phase 3b: Engineering the Main Feature (`X`)

We have successfully built our target dataframe `val_targets_df`, which contains our ground truth `y_cumulative_weight`.

Now, we must create the features (`X`) for each row. Our goal is to create features that "look into the future" from the perspective of our `train_receivals` data, using the `df_po_cleaned` table.

For *each* row in our `val_targets_df` (which is a unique `(rm_id, forecast_end_date)` pair), our model will need to answer: "Based on the purchase orders I know about, what cumulative weight do I *expect* to arrive for this `rm_id` by this `forecast_end_date`?"

Our first and most powerful feature will be **`f_cumulative_po_quantity`**:
* For a given `rm_id` and `forecast_end_date`, this feature will be the `sum` of all `quantity` from `df_po_cleaned` for that `rm_id` where the `delivery_date` is on or before the `forecast_end_date`.

This will be our model's main baseline feature. We will then add other features (like supplier lag, historical receivals, etc.) to help it *adjust* this baseline.

In [4]:
print("--- Phase 3b: Engineering feature 'f_cumulative_po_quantity' ---")

# --- 1. Pre-aggregate POs by rm_id and date ---
# This gives us the total *expected* quantity for each material on each day
# BUGS FIXED:
# 1. 'rm_id' now exists in df_po_cleaned from our new setup cell
# 2. 'groupby' syntax is now a clean list of column names
daily_po_quantity = df_po_cleaned.groupby(
    ['rm_id', 'delivery_date']
)['quantity'].sum().reset_index(name='daily_po_quantity')

# We rename 'delivery_date' to match 'forecast_end_date' for the upcoming merge
daily_po_quantity.rename(columns={'delivery_date': 'po_delivery_date'}, inplace=True)


# --- 2. Merge our target grid with the aggregated POs (on rm_id only) ---
rm_ids_to_merge = val_targets_df['rm_id'].unique()

merged_df = pd.merge(
    val_targets_df[['rm_id', 'forecast_end_date']], # Our "grid"
    daily_po_quantity[daily_po_quantity['rm_id'].isin(rm_ids_to_merge)], # Our "data"
    on='rm_id',
    how='left' # Keep all (rm_id, date) pairs, even if they have no POs
)

# --- 3. Apply the time-filter ---
# For each (rm_id, forecast_end_date) pair, we only keep the POs
# that were "due" on or before that forecast_end_date.
merged_df_filtered = merged_df[
    merged_df['forecast_end_date'] >= merged_df['po_delivery_date']
].copy()

# --- 4. Group by our original grid and sum the quantities ---
# This calculates the cumulative sum.
f_cumulative_po = merged_df_filtered.groupby(
    ['rm_id', 'forecast_end_date']
)['daily_po_quantity'].sum().reset_index(name='f_cumulative_po_quantity')

# --- 5. Merge the new feature back into our main val_targets_df ---
print("Merging new feature into 'val_targets_df'...")
val_targets_df = pd.merge(
    val_targets_df,
    f_cumulative_po,
    on=['rm_id', 'forecast_end_date'],
    how='left'
)

# Fill 0s for (rm_id, date) pairs that had no POs at all
val_targets_df['f_cumulative_po_quantity'] = val_targets_df['f_cumulative_po_quantity'].fillna(0)


# --- 6. Final Checks ---
print("\n--- Final Check of Feature vs. Target ---")
print(val_targets_df.head())

# Let's check the same rm_id as before to see the feature in action
non_zero_rm_id = val_targets_df[val_targets_df['y_cumulative_weight'] > 0]['rm_id'].iloc[0]
print(f"\nExample: rm_id {non_zero_rm_id}")
print(val_targets_df[val_targets_df['rm_id'] == non_zero_rm_id].tail())

print("\nFeature 'f_cumulative_po_quantity' created successfully.")

--- Phase 3b: Engineering feature 'f_cumulative_po_quantity' ---
Merging new feature into 'val_targets_df'...

--- Final Check of Feature vs. Target ---
    rm_id         forecast_end_date  daily_net_weight  y_cumulative_weight  \
0  2123.0 2024-08-01 00:00:00+00:00               0.0                  0.0   
1  2123.0 2024-08-02 00:00:00+00:00               0.0                  0.0   
2  2123.0 2024-08-03 00:00:00+00:00               0.0                  0.0   
3  2123.0 2024-08-04 00:00:00+00:00               0.0                  0.0   
4  2123.0 2024-08-05 00:00:00+00:00               0.0                  0.0   

   f_cumulative_po_quantity  
0                       0.0  
1                       0.0  
2                       0.0  
3                       0.0  
4                       0.0  

Example: rm_id 2123.0
      rm_id         forecast_end_date  daily_net_weight  y_cumulative_weight  \
136  2123.0 2024-12-15 00:00:00+00:00               0.0              12500.0   
137  2123.0 202

## Phase 3c: Engineering Time-Based Features

We have our "ground truth" `y` and our first naive feature `X1` (`f_cumulative_po_quantity`). We saw that `X1` can be 0 while `y` is 12,500, which confirms that we need more features.

The model needs to learn to correct this naive PO forecast. The next logical features to add are simple, time-based features derived from the `forecast_end_date`. This will allow the model to learn patterns like "deliveries are more likely at the end of the month" or "we are forecasting for a day in December."

We will add the following features:
* `f_month`
* `f_day_of_week`
* `f_day_of_year`
* `f_is_month_end`: (Binary flag) Is this date near the end of the month? This is based on the documentation hint.

In [5]:
print("--- Phase 3c: Engineering Time-Based Features ---")

# We will add these new features directly to our 'val_targets_df'

# --- 1. Extract basic date components ---
print("Extracting date components...")
val_targets_df['f_month'] = val_targets_df['forecast_end_date'].dt.month
val_targets_df['f_day_of_week'] = val_targets_df['forecast_end_date'].dt.dayofweek
val_targets_df['f_day_of_year'] = val_targets_df['forecast_end_date'].dt.dayofyear

# --- 2. Create 'is_month_end' feature ---
# This is a boolean (True/False) feature
print("Creating 'f_is_month_end'...")
val_targets_df['f_is_month_end'] = val_targets_df['forecast_end_date'].dt.is_month_end

# --- 3. Final Check ---
print("\n--- Final Check: New Time-Based Features ---")
# Print the head, but now with our new features
print(val_targets_df.head())

# Show a sample from the end of a month
print("\nExample: End of August 2024")
print(val_targets_df[
    (val_targets_df['rm_id'] == val_targets_df['rm_id'].iloc[0]) &
    (val_targets_df['forecast_end_date'] >= '2024-08-30')
].head(2))

print("\nTime-based features created successfully.")

--- Phase 3c: Engineering Time-Based Features ---
Extracting date components...
Creating 'f_is_month_end'...

--- Final Check: New Time-Based Features ---
    rm_id         forecast_end_date  daily_net_weight  y_cumulative_weight  \
0  2123.0 2024-08-01 00:00:00+00:00               0.0                  0.0   
1  2123.0 2024-08-02 00:00:00+00:00               0.0                  0.0   
2  2123.0 2024-08-03 00:00:00+00:00               0.0                  0.0   
3  2123.0 2024-08-04 00:00:00+00:00               0.0                  0.0   
4  2123.0 2024-08-05 00:00:00+00:00               0.0                  0.0   

   f_cumulative_po_quantity  f_month  f_day_of_week  f_day_of_year  \
0                       0.0        8              3            214   
1                       0.0        8              4            215   
2                       0.0        8              5            216   
3                       0.0        8              6            217   
4                       0.

## Phase 3d: Engineering Entity & Lag Features

We will now create our most powerful features, which are based on our key EDA findings:

1.  **Median Lag per `rm_id`**: Our EDA showed that `delivery_lag_days` (Actual Arrival - PO Deadline) is a critical value. We will compute the *median* lag for each `rm_id` from our training data. This will give the model a powerful, learned heuristic (e.g., "rm_id 365 typically arrives 15 days before its deadline").
2.  **Historical Receival Lags**: We'll add features for the total `net_weight` received in the *recent past* (e.g., last 30, 90, and 180 days). This captures recent trends and seasonality.

These features are "data-hungry" and require a careful merge against our `train_receivals` data.

In [6]:
print("--- Phase 3d: Engineering Entity & Lag Features ---")

# --- 1. Calculate Median Lag per rm_id ---
print("Calculating median lag per rm_id from training data...")
# We merge train_receivals with POs to link arrival dates to delivery (deadline) dates
# We use the 'rm_id' we added to df_po_cleaned.
df_train_merged_eda = pd.merge(
    train_receivals,
    df_po_cleaned[['purchase_order_id', 'purchase_order_item_no', 'delivery_date', 'rm_id']],
    on=['purchase_order_id', 'purchase_order_item_no', 'rm_id'],
    how='inner'
)

# Calculate the lag
df_train_merged_eda['delivery_lag_days'] = (
    df_train_merged_eda['date_arrival'] - df_train_merged_eda['delivery_date']
).dt.days

# Create the feature map: (rm_id) -> (median_lag)
rm_id_lag_map = df_train_merged_eda.groupby('rm_id')['delivery_lag_days'].median().reset_index(name='f_median_lag_days')

# Merge this new feature into our validation set
val_targets_df = pd.merge(
    val_targets_df,
    rm_id_lag_map,
    on='rm_id',
    how='left'
)

# Fill NaNs with 0 (for rm_ids in validation that had no training data)
val_targets_df['f_median_lag_days'] = val_targets_df['f_median_lag_days'].fillna(0)

print("Feature 'f_median_lag_days' created.")


# --- 2. Calculate Historical Receival "Lag" Features ---
print("Calculating historical receival features (e.g., last 30 days)...")
# This is a complex operation. We'll pre-calculate daily receivals
# from our *training* data (train_receivals)
daily_receivals_hist = train_receivals.groupby(
    ['rm_id', train_receivals['date_arrival'].dt.date]
)['net_weight'].sum().reset_index()
daily_receivals_hist.rename(columns={'date_arrival': 'date'}, inplace=True)
daily_receivals_hist['date'] = pd.to_datetime(daily_receivals_hist['date'], utc=True)

# Set index for fast lookups
daily_receivals_hist = daily_receivals_hist.set_index(['rm_id', 'date']).sort_index()

# We need a function to apply to each row of our val_targets_df
def calculate_historical_receivals(row, daily_data, days):
    """
    For a given row (rm_id, forecast_end_date), sum receivals
    in the 'days' period *before* that date.
    """
    rm_id = row['rm_id']
    end_date = row['forecast_end_date']
    # The historical period starts 'days' before the validation start date
    # Our validation starts on 2024-08-01. We are calculating
    # the receivals in the (end_date - days) to (end_date - 1) period.
    # This must be done from the *training* data.
    
    # We'll simplify: we'll find the total receivals in the training data
    # in the N days *before our entire validation period starts*.
    # This is a "static" feature, as rolling features are very slow.
    
    # Let's redefine. We'll calculate total receivals in the N days
    # *before* the validation period (i.e., before 2024-08-01)
    
    hist_end_date = pd.to_datetime('2024-08-01', utc=True)
    hist_start_date = hist_end_date - pd.Timedelta(days=days)
    
    try:
        # Get the slice of data for this rm_id in the window
        hist_slice = daily_data.loc[(rm_id, hist_start_date):(rm_id, hist_end_date - pd.Timedelta(days=1))]
        return hist_slice['net_weight'].sum()
    except KeyError:
        return 0.0 # No receivals for this rm_id in that period

# --- 2b. This is very slow. Let's use a faster, map-based approach ---
print("Using faster map-based approach for historical features...")
hist_end_date = pd.to_datetime('2024-08-01', utc=True)
hist_windows = [30, 90, 180]
hist_features = {}

# We only use 'train_receivals' to build these features
for days in hist_windows:
    hist_start_date = hist_end_date - pd.Timedelta(days=days)
    
    # Filter training data to this window
    window_data = train_receivals[
        (train_receivals['date_arrival'] >= hist_start_date) &
        (train_receivals['date_arrival'] < hist_end_date)
    ]
    
    # Group by rm_id and sum
    feature_map = window_data.groupby('rm_id')['net_weight'].sum().reset_index(name=f'f_receivals_{days}d')
    hist_features[days] = feature_map

# Merge all historical features into our validation set
for days in hist_windows:
    val_targets_df = pd.merge(
        val_targets_df,
        hist_features[days],
        on='rm_id',
        how='left'
    )
    # Fill NaNs with 0 (for rm_ids that had no receivals in this window)
    val_targets_df[f'f_receivals_{days}d'] = val_targets_df[f'f_receivals_{days}d'].fillna(0)
    print(f"Feature 'f_receivals_{days}d' created.")


# --- 3. Final Check ---
print("\n--- Final Check: All Features ---")
print(val_targets_df.info())
print("\n--- Example row (head) ---")
print(val_targets_df.head())
print("\n--- Example row (tail) ---")
print(val_targets_df.tail())

print("\nAll features for validation set created successfully.")

--- Phase 3d: Engineering Entity & Lag Features ---
Calculating median lag per rm_id from training data...
Feature 'f_median_lag_days' created.
Calculating historical receival features (e.g., last 30 days)...
Using faster map-based approach for historical features...
Feature 'f_receivals_30d' created.
Feature 'f_receivals_90d' created.
Feature 'f_receivals_180d' created.

--- Final Check: All Features ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7050 entries, 0 to 7049
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   rm_id                     7050 non-null   float64            
 1   forecast_end_date         7050 non-null   datetime64[ns, UTC]
 2   daily_net_weight          7050 non-null   float64            
 3   y_cumulative_weight       7050 non-null   float64            
 4   f_cumulative_po_quantity  7050 non-null   float64            
 5   f_month   

## Phase 3e: Debugging Our Features (The "Cold Start" Problem)

Our last step showed that our features are all 0.0, even when the target `y` is non-zero (e.g., 48,180.0). This is a critical failure.

We must now test our hypothesis: Are the `rm_id`s in our validation set *new*? Or is our feature logic buggy?

This cell will perform a check to see how many of the `rm_id`s in our validation set (`val_targets_df`) also have *any* history in our training set (`train_receivals`).

In [7]:
print("--- Phase 3e: Debugging Feature 'Cold Start' ---")

# --- 1. Get our two sets of rm_ids ---
validation_rm_ids = set(val_targets_df['rm_id'].unique())
training_rm_ids = set(train_receivals['rm_id'].unique())

print(f"Total unique rm_ids in Validation Set: {len(validation_rm_ids)}")
print(f"Total unique rm_ids in Training Set:   {len(training_rm_ids)}")

# --- 2. Find the intersection (rm_ids in BOTH sets) ---
rm_ids_in_both = validation_rm_ids.intersection(training_rm_ids)
print(f"\nrm_ids that are in BOTH sets (the overlap): {len(rm_ids_in_both)}")

# --- 3. Find the rm_ids ONLY in the validation set ---
rm_ids_only_in_validation = validation_rm_ids.difference(training_rm_ids)
print(f"rm_ids that are ONLY in the Validation Set (Cold Start): {len(rm_ids_only_in_validation)}")

# --- 4. Manually check one of our problem rm_ids ---
rm_id_to_check = 4501.0
is_in_training = rm_id_to_check in training_rm_ids
print(f"\n--- Manual Check ---")
print(f"Is our problem rm_id {rm_id_to_check} in the training set?  {is_in_training}")

if is_in_training:
    print("\nDEBUG: The rm_id IS in the training set. This means our feature logic is buggy.")
    # Let's see if our logic was right, but the window was empty
    hist_end_date = pd.to_datetime('2024-08-01', utc=True)
    hist_start_date = hist_end_date - pd.Timedelta(days=180)
    
    print(f"Checking for receivals for {rm_id_to_check} between {hist_start_date.date()} and {hist_end_date.date()}...")
    
    rm_id_history = train_receivals[
        (train_receivals['rm_id'] == rm_id_to_check) &
        (train_receivals['date_arrival'] >= hist_start_date) &
        (train_receivals['date_arrival'] < hist_end_date)
    ]
    print(f"Total receivals found in window: {len(rm_id_history)}")
    print(f"Total net_weight in window: {rm_id_history['net_weight'].sum()}")
    
else:
    print("\nDEBUG: The rm_id is NOT in the training set. This is a 'Cold Start' problem.")
    print("This confirms our features are 0.0 because there is no history to find.")

print("\nDebug check complete.")

--- Phase 3e: Debugging Feature 'Cold Start' ---
Total unique rm_ids in Validation Set: 50
Total unique rm_ids in Training Set:   195

rm_ids that are in BOTH sets (the overlap): 42
rm_ids that are ONLY in the Validation Set (Cold Start): 8

--- Manual Check ---
Is our problem rm_id 4501.0 in the training set?  False

DEBUG: The rm_id is NOT in the training set. This is a 'Cold Start' problem.
This confirms our features are 0.0 because there is no history to find.

Debug check complete.


## Phase 3f: Saving the Model-Ready Dataset

We have successfully built our complete validation dataset. The debug in the previous step confirms that our "all zero" features were not a bug, but a correct representation of the "cold start" problem for 8 of our 50 validation `rm_id`s.

Our dataframe `val_targets_df` is now complete. It contains:
* **The Target (`y`):** `y_cumulative_weight`
* **The Features (`X`):** All columns starting with `f_`

We will now save this dataframe to a `.parquet` file. This is a fast, efficient file format that will allow all our future "model" notebooks to load this data instantly, without re-running all the feature engineering steps.

In [8]:
print("--- Phase 3f: Saving Final Validation Dataset ---")

# Define our feature names
# We don't want to use the 'y' or 'daily_net_weight' as features
feature_columns = [col for col in val_targets_df.columns if col.startswith('f_')]
target_column = 'y_cumulative_weight'

# Let's also save the key "identifier" columns
id_columns = ['rm_id', 'forecast_end_date']

# Define the final dataframe to save
# This has our IDs, our features, and our target.
final_validation_set = val_targets_df[id_columns + feature_columns + [target_column]].copy()

# --- Save the file ---
FILE_PATH = "validation_dataset.parquet"
try:
    final_validation_set.to_parquet(FILE_PATH, index=False)
    print(f"\nSuccessfully saved model-ready dataset to '{FILE_PATH}'")
    print("\nColumns in saved file:")
    print(final_validation_set.info())
    
    print("\n--- 02_Feature_Engineering.ipynb Complete ---")
    
except Exception as e:
    print(f"\nError saving file: {e}")
    print("You may need to install 'pyarrow' or 'fastparquet':")
    print("pip install pyarrow")

--- Phase 3f: Saving Final Validation Dataset ---

Successfully saved model-ready dataset to 'validation_dataset.parquet'

Columns in saved file:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7050 entries, 0 to 7049
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   rm_id                     7050 non-null   float64            
 1   forecast_end_date         7050 non-null   datetime64[ns, UTC]
 2   f_cumulative_po_quantity  7050 non-null   float64            
 3   f_month                   7050 non-null   int32              
 4   f_day_of_week             7050 non-null   int32              
 5   f_day_of_year             7050 non-null   int32              
 6   f_is_month_end            7050 non-null   bool               
 7   f_median_lag_days         7050 non-null   float64            
 8   f_receivals_30d           7050 non-null   float64            
 9   f_rec

---
# Phase 4: Building the Training Dataset (Multi-Year)

We have successfully built our `validation_dataset.parquet` (for Aug-Dec 2024).

We now need a `training_dataset.parquet` to fit our model on. Based on our discussion, simply using 2023 is not enough. We will create a much more robust training set by combining several historical time blocks.

Our plan is to create a function that can generate a complete `(X, y)` feature set for *any* given time period. We will then run this function for multiple recent years (e.g., 2023, 2022, 2021) and concatenate the results.

This gives our model more data to learn from while perfectly respecting the time-series nature of the problem and protecting against concept drift.

In [10]:
print("--- Phase 4: Building Multi-Year Training Set ---")

def create_feature_set(start_date_str, end_date_str):
    """
    Generates a complete (X, y) feature set for the given date range.
    All features are calculated using data *before* the start_date_str.
    """
    print(f"\n--- Generating feature set for {start_date_str} to {end_date_str} ---")
    
    # --- 1. Define Time Universe ---
    TRAIN_START_DATE = pd.to_datetime(start_date_str, utc=True)
    TRAIN_END_DATE = pd.to_datetime(end_date_str, utc=True)
    
    # Use *all* rm_ids from our full history for the universe
    all_rm_ids = df_receivals_cleaned['rm_id'].unique()
    train_dates = pd.date_range(start=TRAIN_START_DATE, end=TRAIN_END_DATE, freq='D', tz='UTC')
    
    # Create the (rm_id, date) grid
    train_universe = list(itertools.product(all_rm_ids, train_dates))
    df = pd.DataFrame(train_universe, columns=['rm_id', 'forecast_end_date'])
    
    # --- 2. Define Historical Data for this Block ---
    # This is all receival data *before* this block's start date
    hist_receivals = df_receivals_cleaned[df_receivals_cleaned['date_arrival'] < TRAIN_START_DATE].copy()
    # This is all PO data (we'll filter POs by date later)
    hist_po = df_po_cleaned.copy()

    # --- 3. Build Target (y_cumulative_weight) ---
    print("Building target (y)...")
    daily_actuals = df_receivals_cleaned[
        (df_receivals_cleaned['date_arrival'] >= TRAIN_START_DATE) &
        (df_receivals_cleaned['date_arrival'] <= TRAIN_END_DATE)
    ]
    daily_actuals_grouped = daily_actuals.groupby(
        ['rm_id', daily_actuals['date_arrival'].dt.date]
    )['net_weight'].sum().reset_index(name='daily_net_weight')
    daily_actuals_grouped['forecast_end_date'] = pd.to_datetime(daily_actuals_grouped['date_arrival'], utc=True)
    
    df = pd.merge(
        df,
        daily_actuals_grouped[['rm_id', 'forecast_end_date', 'daily_net_weight']],
        on=['rm_id', 'forecast_end_date'],
        how='left'
    )
    df['daily_net_weight'] = df['daily_net_weight'].fillna(0)
    
    df = df.sort_values(by=['rm_id', 'forecast_end_date'])
    df['y_cumulative_weight'] = df.groupby('rm_id')['daily_net_weight'].cumsum()
    
    # --- 4. Build Feature (f_cumulative_po_quantity) ---
    print("Building feature (f_cumulative_po_quantity)...")
    daily_po_quantity = hist_po.groupby(
        ['rm_id', 'delivery_date']
    )['quantity'].sum().reset_index(name='daily_po_quantity')
    daily_po_quantity.rename(columns={'delivery_date': 'po_delivery_date'}, inplace=True)
    
    rm_ids_to_merge = df['rm_id'].unique()
    merged_df = pd.merge(
        df[['rm_id', 'forecast_end_date']],
        daily_po_quantity[daily_po_quantity['rm_id'].isin(rm_ids_to_merge)],
        on='rm_id',
        how='left'
    )
    merged_df_filtered = merged_df[merged_df['forecast_end_date'] >= merged_df['po_delivery_date']].copy()
    f_cumulative_po = merged_df_filtered.groupby(
        ['rm_id', 'forecast_end_date']
    )['daily_po_quantity'].sum().reset_index(name='f_cumulative_po_quantity')
    
    df = pd.merge(df, f_cumulative_po, on=['rm_id', 'forecast_end_date'], how='left')
    df['f_cumulative_po_quantity'] = df['f_cumulative_po_quantity'].fillna(0)
    
    # --- 5. Build Features (Time-Based) ---
    print("Building features (Time-Based)...")
    df['f_month'] = df['forecast_end_date'].dt.month
    df['f_day_of_week'] = df['forecast_end_date'].dt.dayofweek
    df['f_day_of_year'] = df['forecast_end_date'].dt.dayofyear
    df['f_is_month_end'] = df['forecast_end_date'].dt.is_month_end

    # --- 6. Build Features (Entity & Lag) ---
    print("Building features (Entity & Lag)...")
    
    # --- f_median_lag_days ---
    df_train_merged_eda = pd.merge(
        hist_receivals,
        hist_po[['purchase_order_id', 'purchase_order_item_no', 'delivery_date', 'rm_id']],
        on=['purchase_order_id', 'purchase_order_item_no', 'rm_id'],
        how='inner'
    )
    df_train_merged_eda['delivery_lag_days'] = (
        df_train_merged_eda['date_arrival'] - df_train_merged_eda['delivery_date']
    ).dt.days
    rm_id_lag_map = df_train_merged_eda.groupby('rm_id')['delivery_lag_days'].median().reset_index(name='f_median_lag_days')
    df = pd.merge(df, rm_id_lag_map, on='rm_id', how='left')
    df['f_median_lag_days'] = df['f_median_lag_days'].fillna(0)

    # --- f_receivals_Nd ---
    hist_windows = [30, 90, 180]
    for days in hist_windows:
        hist_start_date_window = TRAIN_START_DATE - pd.Timedelta(days=days)
        window_data = hist_receivals[
            (hist_receivals['date_arrival'] >= hist_start_date_window) &
            (hist_receivals['date_arrival'] < TRAIN_START_DATE)
        ]
        feature_map = window_data.groupby('rm_id')['net_weight'].sum().reset_index(name=f'f_receivals_{days}d')
        df = pd.merge(df, feature_map, on='rm_id', how='left')
        df[f'f_receivals_{days}d'] = df[f'f_receivals_{days}d'].fillna(0)
    
    print(f"--- Block for {start_date_str} complete. Shape: {df.shape} ---")
    
    # Filter out rows where the cumulative target is always 0 (no activity)
    # This keeps our training set focused on active rm_ids for that period
    rm_id_max_y = df.groupby('rm_id')['y_cumulative_weight'].max()
    active_rm_ids = rm_id_max_y[rm_id_max_y > 0].index
    df = df[df['rm_id'].isin(active_rm_ids)]
    
    print(f"--- Filtered to active rm_ids. Final Shape: {df.shape} ---")
    return df


# --- Now, let's build our training set using this function ---
print("\n\n--- Building Full Training Set ---")
# We'll use 4 blocks, going back to 2020
years = [2023, 2022, 2021, 2020]
training_blocks = []

for year in years:
    start_date = f'{year}-08-01'
    end_date = f'{year}-12-31'
    
    # --- FIX IS HERE ---
    # We must make our comparison timestamp (f'{year}-01-01') timezone-aware (UTC)
    # to match train_receivals['date_arrival'].min()
    if pd.to_datetime(f'{year}-01-01', utc=True) > train_receivals['date_arrival'].min():
        block = create_feature_set(start_date, end_date)
        training_blocks.append(block)

# Concatenate all blocks into one dataset
df_train = pd.concat(training_blocks, ignore_index=True)

print("\n\n--- Full Training Set Created ---")
print(df_train.info())


# --- Save the file ---
FILE_PATH = "training_dataset.parquet"
try:
    # Define our feature names
    feature_columns = [col for col in df_train.columns if col.startswith('f_')]
    target_column = 'y_cumulative_weight'
    id_columns = ['rm_id', 'forecast_end_date']

    # Define the final dataframe to save
    final_training_set = df_train[id_columns + feature_columns + [target_column]].copy()
    
    final_training_set.to_parquet(FILE_PATH, index=False)
    print(f"\nSuccessfully saved model-ready dataset to '{FILE_PATH}'")
    
except Exception as e:
    print(f"\nError saving file: {e}")

--- Phase 4: Building Multi-Year Training Set ---


--- Building Full Training Set ---

--- Generating feature set for 2023-08-01 to 2023-12-31 ---
Building target (y)...
Building feature (f_cumulative_po_quantity)...
Building features (Time-Based)...
Building features (Entity & Lag)...
--- Block for 2023-08-01 complete. Shape: (31059, 13) ---
--- Filtered to active rm_ids. Final Shape: (7038, 13) ---

--- Generating feature set for 2022-08-01 to 2022-12-31 ---
Building target (y)...
Building feature (f_cumulative_po_quantity)...
Building features (Time-Based)...
Building features (Entity & Lag)...
--- Block for 2022-08-01 complete. Shape: (31059, 13) ---
--- Filtered to active rm_ids. Final Shape: (5508, 13) ---

--- Generating feature set for 2021-08-01 to 2021-12-31 ---
Building target (y)...
Building feature (f_cumulative_po_quantity)...
Building features (Time-Based)...
Building features (Entity & Lag)...
--- Block for 2021-08-01 complete. Shape: (31059, 13) ---
--- Filtered to ac